In [1]:
import sys
import os
sys.path.append(".")  
import numpy as np
import pandas as pd
from tqdm import tqdm
import aiohttp
import asyncio
from PIL import Image

from src.utils import load_image_from_url
from models.embedder import embed_images_batch

c:\Users\HP\Desktop\image-search\.venv\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [2]:
df = pd.read_csv(
    "data/unsplash_lite/photos.csv",
    sep="\t",
    on_bad_lines="skip"
)

df = df.dropna(subset=["photo_image_url"])
df = df[df["photo_image_url"].str.startswith("https")]
df = df.reset_index(drop=True)

print("Total usable:", len(df))

Total usable: 25000


In [3]:
async def run_batched(df, batch_size=32):

    embeddings = []
    ids = []

    connector = aiohttp.TCPConnector(limit=20)
    timeout = aiohttp.ClientTimeout(total=10)

    batch_images = []
    batch_ids = []

    async with aiohttp.ClientSession(connector=connector, timeout=timeout) as session:

        for _, row in tqdm(df.iterrows(), total=len(df)):

            try:
                img = await load_image_from_url(session, row["photo_image_url"])
            except:
                continue

            if img is None:
                continue

            batch_images.append(img)
            batch_ids.append(row["photo_id"])

            if len(batch_images) >= batch_size:
                vecs = embed_images_batch(batch_images)

                embeddings.append(vecs)
                ids.extend(batch_ids)

                batch_images.clear()
                batch_ids.clear()

        # flush remaining
        if batch_images:
            vecs = embed_images_batch(batch_images)
            embeddings.append(vecs)
            ids.extend(batch_ids)

    embeddings = np.concatenate(embeddings, axis=0).astype("float32")

    return ids, embeddings

In [4]:
ids_test, emb_test = await run_batched(df.iloc[:50], batch_size=16)

print(len(ids_test), emb_test.shape)

100%|██████████| 50/50 [00:40<00:00,  1.24it/s]


50 (50, 768)


In [5]:
ids, embeddings = await run_batched(df, batch_size=32)

  0%|          | 0/25000 [00:00<?, ?it/s]

  3%|▎         | 665/25000 [10:28<4:30:48,  1.50it/s] c:\Users\HP\Desktop\image-search\.venv\Lib\site-packages\PIL\Image.py:3451: DecompressionBombWarning: Image size (96012000 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
  5%|▌         | 1295/25000 [20:55<2:52:26,  2.29it/s] c:\Users\HP\Desktop\image-search\.venv\Lib\site-packages\PIL\Image.py:3451: DecompressionBombWarning: Image size (99996120 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
 10%|▉         | 2413/25000 [40:14<6:09:50,  1.02it/s] c:\Users\HP\Desktop\image-search\.venv\Lib\site-packages\PIL\Image.py:3451: DecompressionBombWarning: Image size (107184040 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
 14%|█▍        | 3441/25000 [56:02<3:02:01,  1.97it/s] c:\Users\HP\Desktop\image-search\.venv\Lib\site-packages\PIL\Image.py:3451: DecompressionBombWarning: Image size (

In [6]:
print(embeddings.shape, len(ids))

import numpy as np
print("NaNs:", np.isnan(embeddings).any())

(15995, 768) 15995
NaNs: False


In [7]:
embeddings = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)

In [8]:
import faiss

d = embeddings.shape[1]

index = faiss.IndexHNSWFlat(d, 32)
index.hnsw.efConstruction = 200

index.add(embeddings)
index.hnsw.efSearch = 50

print("Index size:", index.ntotal)

Index size: 15995


In [9]:
np.save("data/unsplash/embeddings_25k.npy", embeddings)
np.save("data/unsplash/ids_25k.npy", np.array(ids))

faiss.write_index(index, "data/unsplash/index_hnsw_25k.faiss")